In [ ]:
# 必要なパッケージの読み込み
library(caret)
library(Metrics)
library(ggplot2)
library(lattice)

# 対象ファイル名一覧
file_names <- c(
  "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
  "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
  "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
  "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)

# データパスと日付
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
today <- format(Sys.Date(), "%Y%m%d")

# 評価指標の初期化
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_C", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features")
result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

# メイン処理ループ
for (file in file_names) {
  filepath <- paste0(base_path, file)
  cat("\n=== Processing:", file, "===\n")
  df_all <- read.csv(filepath, row.names = 1)
  cat("Final dataset size:", dim(df_all)[1], dim(df_all)[2], "\n")

  feature_vars <- setdiff(colnames(df_all), target_vars)

  for (target_var in target_vars) {
    cat("\n---\nTraining model for:", target_var, "on", file, "\n")
    df <- df_all[, c(feature_vars, target_var)]
    df <- df[complete.cases(df), ]

    # モデルチューニング設定
    tune_grid <- expand.grid(C = 2^(-5:5))
    ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")

      
      
    model <- train(
      formula(paste(target_var, "~ .")),
      data = df,
      method = "svmLinear",
      metric = "RMSE",
      trControl = ctrl,
      tuneGrid = tune_grid
    )

    # 交差検証予測値の取得とフィルタリング（安全に）
    cv_preds <- model$pred
    best_params <- model$bestTune

    # bestTuneのパラメータだけ残っている列でフィルタ
    for (param in names(best_params)) {
      if (param %in% colnames(cv_preds)) {
        cv_preds <- cv_preds[cv_preds[[param]] == best_params[[param]], ]
      }
    }

    # 値が空でないか確認
    if (nrow(cv_preds) > 0) {
      pred <- cv_preds$pred
      obs <- cv_preds$obs

      R2 <- round(cor(obs, pred)^2, 3)
      RMSE_val <- round(rmse(obs, pred), 3)
      MAE_val <- round(mae(obs, pred), 3)
      RPD_val <- round(sd(obs) / RMSE_val, 3)
    } else {
      warning("No predictions matched bestTune. Skipping this model evaluation.")
      R2 <- RMSE_val <- MAE_val <- RPD_val <- NA
    }


    cat("Best C value:", model$bestTune$C, "\n")
    cat(file, target_var, ": R2 =", R2, ", RMSE =", RMSE_val, ", MAE =", MAE_val, ", RPD =", RPD_val, "\n")

    # 結果保存
    result_matrix[paste0("Best_C_", target_var), file] <- model$bestTune$C
    result_matrix[paste0("R2_", target_var), file] <- R2
    result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
    result_matrix[paste0("MAE_", target_var), file] <- MAE_val
    result_matrix[paste0("RPD_", target_var), file] <- RPD_val
    result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
    result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

    # プロット設定
    get_axis_limit <- function(values) {
      max_val <- max(values, na.rm = TRUE)
      if (max_val <= 1.5) return(ceiling(max_val / 0.1) * 0.1)
      if (max_val <= 5) return(ceiling(max_val / 0.5) * 0.5)
      if (max_val <= 30) return(ceiling(max_val / 2) * 2)
      return(ceiling(max_val / 5) * 5)
    }
    range_max <- get_axis_limit(c(obs, pred))

    # プロット生成
    p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
      geom_point(color = "blue", alpha = 0.7) +
      geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
      scale_x_continuous(limits = c(0, range_max)) +
      scale_y_continuous(limits = c(0, range_max)) +
      coord_fixed(ratio = 1) +
      labs(
        title = paste0(target_var, " Prediction (", file, ")"),
        x = "Observed", y = "Predicted"
      ) +
      theme_minimal() +
      annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
               label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
      annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
               label = paste0("R² = ", R2))

    # モデル保存
    model_data_bundle <- list(model = model, training_data = df)
    rds_file <- paste0("new20250702_svmLinear_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_svmLinear_", today, ".rds")
    saveRDS(model_data_bundle, file = rds_file)
    
    # --- 変数重要度の保存 ---
    # --- 変数重要度の保存（エラー対策付き） ---
    imp_obj <- varImp(model, scale = TRUE)
    importance <- data.frame(Variable = rownames(imp_obj$importance),
                             Importance = imp_obj$importance[, 1])  # 1列目を抽出

    # 保存
    imp_file <- paste0("new20250702_importance_", tools::file_path_sans_ext(file), "_", target_var, "_svmLinear_", today, ".csv")
    write.csv(importance, imp_file, row.names = FALSE)


    # プロット保存
    plot_file <- paste0("new20250702_plot_", tools::file_path_sans_ext(file), "_", target_var, "_svmLinear_", today, ".pdf")
    pdf(plot_file, width = 6, height = 6)
    print(p)
    dev.off()
  }
}

# 結果の保存
output_file <- paste0("new20250702_svmLinear_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\nSummary saved as", output_file, "\n")



=== Processing: n_base.csv ===
Final dataset size: 358 11 

---
Training model for: Jsc on n_base.csv 
Best C value: 0.03125 
n_base.csv Jsc : R2 = 0.242 , RMSE = 4.399 , MAE = 3.574 , RPD = 1.15 

---
Training model for: Voc on n_base.csv 
Best C value: 0.03125 
n_base.csv Voc : R2 = 0.708 , RMSE = 0.082 , MAE = 0.059 , RPD = 1.849 

---
Training model for: FF on n_base.csv 
Best C value: 0.03125 
n_base.csv FF : R2 = 0.122 , RMSE = 0.105 , MAE = 0.083 , RPD = 1.062 

---
Training model for: PCEmax on n_base.csv 
Best C value: 0.125 
n_base.csv PCEmax : R2 = 0.249 , RMSE = 2.303 , MAE = 1.809 , RPD = 1.148 


Warning message:
"Removed 1 row containing missing values or values outside the scale range
(`geom_point()`)."



=== Processing: n_base_OH.csv ===
Final dataset size: 358 142 

---
Training model for: Jsc on n_base_OH.csv 


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."


Best C value: 0.0625 
n_base_OH.csv Jsc : R2 = 0.739 , RMSE = 2.611 , MAE = 1.933 , RPD = 1.938 

---
Training model for: Voc on n_base_OH.csv 


Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x,

Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
